# Credit Applications Data Pipeline

This notebook covers the full data pipeline for the raw credit applications dataset — from ingestion through cleaning and export.

**Pipeline stages:**
1. Load raw JSON
2. Flatten nested structure → CSV
3. Understand the dataset (shape, types, missing values)
4. Clean and validate all fields
5. Export cleaned CSV

---

## 1. Imports

Standard libraries used throughout the pipeline:

| Library | Purpose |
|---|---|
| `json` | Parse the raw JSON input file |
| `os` | Build file paths in an OS-agnostic way |
| `pandas` | Core data manipulation and DataFrame operations |
| `numpy` | Numeric operations and NaN handling |
| `matplotlib` / `seaborn` | Visualisations (used in EDA) |
| `hashlib` | SHA-256 hashing of PII fields |
| `re` | Regular expressions for format validation |

In [45]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import re


---

## 2. Loading the Raw JSON

The source file is a JSON document. Because JSON can be structured either as a **top-level list** (`[{...}, {...}]`) or as a **dict with a list value** (`{"records": [{...}]}`), the loader handles both shapes automatically:

- If the root is a **dict**, it finds the first key whose value is a list and uses that.
- If the root is already a **list**, it is used directly.

This makes the loader robust to minor upstream schema changes.

In [46]:
# ── Config ─────────────────────────────────────────────────────────────────────
INPUT_PATH  = '/workspaces/dego-project-team11/data/' \
             'raw_credit_applications.json'
OUTPUT_PATH = os.path.join(os.path.dirname(INPUT_PATH), 'raw_credit_applications.csv')

# ── Load ───────────────────────────────────────────────────────────────────────
if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(f"JSON file not found: {INPUT_PATH}")

with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    try:
        data = json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON: {e}")

if isinstance(data, dict):
    list_keys = [k for k, v in data.items() if isinstance(v, list)]
    data = data[list_keys[0]] if list_keys else [data]

---

## 3. Flattening JSON → CSV

Each JSON record has a **nested structure** with four main sub-objects:

```
{
  "applicant_info": { ... },
  "financials":     { ... },
  "spending_behavior": [ {"category": "...", "amount": ...}, ... ],
  "decision":       { ... }
}
```

### Flattening strategy

Each sub-object is expanded into **prefixed flat columns** so the resulting CSV is tidy and unambiguous:

| Sub-object | Example output column |
|---|---|
| `applicant_info` | `applicant_info_email`, `applicant_info_dob` |
| `financials` | `financials_annual_income`, `financials_debt_to_income` |
| `spending_behavior` | `spending_behavior_groceries`, `spending_behavior_travel` |
| `decision` | `decision_loan_approved`, `decision_interest_rate` |

### Spending behavior (list → wide columns)

The `spending_behavior` field is a list of `{category, amount}` pairs — one entry per spending category. These are **pivoted to wide format** (one column per category) using a dict comprehension. Category names are lowercased and spaces replaced with underscores for consistent column naming.

### Name splitting

`full_name` is split into `first_name` and `last_name` on the first whitespace. This is a best-effort split — compound surnames or prefixes are kept together in `last_name`.

### Column ordering

Columns are sorted into a logical order: applicant info → financials → spending → decision → timestamp. Spending columns are sorted alphabetically within their group.

In [47]:
# ── Build rows ─────────────────────────────────────────────────────────────────
rows = []
for record in data:

    # 1. applicant_info
    ai = record.get('applicant_info', {})
    full_name  = ai.get('full_name', '')
    name_parts = full_name.strip().split(None, 1)
    applicant_info = {
        'applicant_info_id'         : record.get('_id'),
        'applicant_info_full_name'  : full_name,
        'applicant_info_first_name' : name_parts[0] if len(name_parts) > 0 else '',
        'applicant_info_last_name'  : name_parts[1] if len(name_parts) > 1 else '',
        'applicant_info_email'      : ai.get('email'),
        'applicant_info_ssn'        : ai.get('ssn'),
        'applicant_info_ip_address' : ai.get('ip_address'),
        'applicant_info_gender'     : ai.get('gender'),
        'applicant_info_dob'        : ai.get('date_of_birth'),
        'applicant_info_zip_code'   : ai.get('zip_code'),
    }

    # 2. financials
    fin = record.get('financials', {})
    financials = {
        'financials_annual_income'         : fin.get('annual_income'),
        'financials_credit_history_months' : fin.get('credit_history_months'),
        'financials_debt_to_income'        : fin.get('debt_to_income'),
        'financials_savings_balance'       : fin.get('savings_balance'),
    }

    # 3. spending_behavior — one column per category
    spending_raw = record.get('spending_behavior', [])
    spending = {
        f'spending_behavior_{s["category"].lower().replace(" ", "_")}': s['amount']
        for s in spending_raw
    }

    # 4. decision
    dec = record.get('decision', {})
    decision = {
        'decision_loan_approved'    : dec.get('loan_approved'),
        'decision_rejection_reason' : dec.get('rejection_reason'),
        'decision_interest_rate'    : dec.get('interest_rate'),
        'decision_approved_amount'  : dec.get('approved_amount'),
        'decision_loan_purpose'     : record.get('loan_purpose'),
        'decision_notes'            : record.get('notes'),
    }

    # 5. processing_timestamp
    timestamp = {
        'processing_timestamp': record.get('processing_timestamp'),
    }

    rows.append({**applicant_info, **financials, **spending, **decision, **timestamp})

# ── Build DataFrame & enforce column order ─────────────────────────────────────
df = pd.DataFrame(rows)

spending_cols = sorted([c for c in df.columns if c.startswith('spending_behavior_')])
other_cols    = [c for c in df.columns if not c.startswith('spending_behavior_')]

def cols_starting(prefix):
    return [c for c in other_cols if c.startswith(prefix)]

ordered_cols = (
    cols_starting('applicant_info_')
    + cols_starting('financials_')
    + spending_cols
    + cols_starting('decision_')
    + cols_starting('processing_timestamp')
)

df = df[ordered_cols]

# ── Export ─────────────────────────────────────────────────────────────────────
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')  # utf-8-sig for Excel compatibility
print(f"✓ Saved {len(df)} rows × {len(df.columns)} columns to {OUTPUT_PATH}")

✓ Saved 502 rows × 36 columns to /workspaces/dego-project-team11/data/raw_credit_applications.csv


---

## 4. Understanding the Dataset

Before cleaning, we inspect the dataset to understand its structure, variable types, and data quality issues. This informs every decision in the cleaning phase.

Key things we look for:
- **Shape** — how many records and columns do we have?
- **Data types** — are columns parsed correctly (e.g. dates as strings, numbers as floats)?
- **Descriptive statistics** — what are the ranges, means, and distributions of numerical variables?
- **Missing values** — which fields have gaps and how severe are they?
- **Sentinel values** — the raw data uses `-1` as a placeholder for missing/unknown values; these need to be converted to proper `NaN`.

In [48]:
df = pd.read_csv("/workspaces/dego-project-team11/data/raw_credit_applications.csv")

print("Shape:", df.shape)
display(df.head())

Shape: (502, 36)


,applicant_info_id,applicant_info_full_name,applicant_info_first_name,applicant_info_last_name,applicant_info_email,applicant_info_ssn,applicant_info_ip_address,applicant_info_gender,applicant_info_dob,applicant_info_zip_code,...,spending_behavior_transportation,spending_behavior_travel,spending_behavior_utilities,decision_loan_approved,decision_rejection_reason,decision_interest_rate,decision_approved_amount,decision_loan_purpose,decision_notes,processing_timestamp
0,app_200,Jerry Smith,Jerry,Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,...,NaN,NaN,NaN,False,algorithm_risk_score,NaN,NaN,NaN,NaN,2024-01-15T00:00:00Z
1,app_037,Brandon Walker,Brandon,Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032.0,...,NaN,NaN,NaN,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,Scott Moore,Scott,Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,...,NaN,NaN,NaN,True,NaN,3.7,59000.0,vacation,NaN,NaN
3,app_024,Thomas Lee,Thomas,Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077.0,...,NaN,NaN,NaN,True,NaN,4.3,34000.0,NaN,NaN,NaN
4,app_184,Brian Rodriguez,Brian,Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080.0,...,NaN,NaN,NaN,False,algorithm_risk_score,NaN,NaN,NaN,NaN,2024-01-15T00:00:00Z


In [49]:
print("\nDataFrame Information:")
df.info()


DataFrame Information:
<class 'pandas.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 36 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   applicant_info_id                      502 non-null    str    
 1   applicant_info_full_name               502 non-null    str    
 2   applicant_info_first_name              502 non-null    str    
 3   applicant_info_last_name               502 non-null    str    
 4   applicant_info_email                   495 non-null    str    
 5   applicant_info_ssn                     497 non-null    str    
 6   applicant_info_ip_address              497 non-null    str    
 7   applicant_info_gender                  499 non-null    str    
 8   applicant_info_dob                     497 non-null    str    
 9   applicant_info_zip_code                500 non-null    float64
 10  financials_annual_income               497 non-null    float6

In [50]:
print("\nDescriptive Statistics for Numerical Columns Before Cleaning:")
print(df.describe())

print("\nDescriptive Statistics for Non-Numerical Columns Before Cleaning:")
print(df.describe(include='object'))


Descriptive Statistics for Numerical Columns Before Cleaning:
       applicant_info_zip_code  financials_annual_income  \
count               500.000000                497.000000   
mean              47660.026000              82705.096155   
std               39517.972093              28101.977862   
min               10001.000000                  0.000000   
25%               10048.000000              63000.000000   
50%               10097.500000              81000.000000   
75%               90244.000000             101000.000000   
max               90299.000000             171000.000000   

       financials_credit_history_months  financials_debt_to_income  \
count                        502.000000                 502.000000   
mean                          50.402390                   0.246195   
std                           31.234824                   0.136296   
min                          -10.000000                   0.050000   
25%                           27.250000       

/tmp/ipykernel_2826/3531735966.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include='object'))


In [51]:
# Types of variables

numerical_variables = df.select_dtypes(include=[np.number]).columns.tolist()
print("\nNumerical Variables:", numerical_variables)

categorical_variables = df.select_dtypes(include=['object']).columns.tolist()
print("\nCategorical Variables:", categorical_variables)


Numerical Variables: ['applicant_info_zip_code', 'financials_annual_income', 'financials_credit_history_months', 'financials_debt_to_income', 'financials_savings_balance', 'spending_behavior_adult_entertainment', 'spending_behavior_alcohol', 'spending_behavior_dining', 'spending_behavior_education', 'spending_behavior_entertainment', 'spending_behavior_fitness', 'spending_behavior_gambling', 'spending_behavior_groceries', 'spending_behavior_healthcare', 'spending_behavior_insurance', 'spending_behavior_rent', 'spending_behavior_shopping', 'spending_behavior_transportation', 'spending_behavior_travel', 'spending_behavior_utilities', 'decision_interest_rate', 'decision_approved_amount']

Categorical Variables: ['applicant_info_id', 'applicant_info_full_name', 'applicant_info_first_name', 'applicant_info_last_name', 'applicant_info_email', 'applicant_info_ssn', 'applicant_info_ip_address', 'applicant_info_gender', 'applicant_info_dob', 'decision_rejection_reason', 'decision_loan_purpose'

/tmp/ipykernel_2826/4281120199.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_variables = df.select_dtypes(include=['object']).columns.tolist()


### Handling `-1` sentinel values

The raw dataset uses `-1` as a stand-in for missing or unknown values across several numeric columns. We first count how many `-1` values exist per column, then replace them all with `NaN` so that pandas treats them correctly in all downstream operations (statistics, imputation, type casting, etc.).

After replacement, a missing value summary is generated showing both the **count** and **percentage** of nulls per column, sorted by severity.

In [52]:
df_work = df.copy()

# Show -1 counts before replacement
print((df_work == -1).sum().sort_values(ascending=False))

applicant_info_id                        0
applicant_info_full_name                 0
applicant_info_first_name                0
applicant_info_last_name                 0
applicant_info_email                     0
applicant_info_ssn                       0
applicant_info_ip_address                0
applicant_info_gender                    0
applicant_info_dob                       0
applicant_info_zip_code                  0
financials_annual_income                 0
financials_credit_history_months         0
financials_debt_to_income                0
financials_savings_balance               0
spending_behavior_adult_entertainment    0
spending_behavior_alcohol                0
spending_behavior_dining                 0
spending_behavior_education              0
spending_behavior_entertainment          0
spending_behavior_fitness                0
spending_behavior_gambling               0
spending_behavior_groceries              0
spending_behavior_healthcare             0
spending_be

In [53]:
df_work.replace(-1, np.nan, inplace=True)

missing_table = pd.DataFrame({
    "missing_count": df_work.isna().sum(),
    "missing_%": df_work.isna().mean() * 100
})

# Round percentage
missing_table["missing_%"] = missing_table["missing_%"].round(2)

# Sort by highest missing %
missing_table = missing_table.sort_values(by="missing_count", ascending=False)

missing_table

,missing_count,missing_%
decision_notes,500,99.60
spending_behavior_adult_entertainment,497,99.00
spending_behavior_gambling,495,98.61
spending_behavior_alcohol,491,97.81
decision_loan_purpose,452,90.04
spending_behavior_shopping,448,89.24
spending_behavior_rent,443,88.25
spending_behavior_transportation,441,87.85
processing_timestamp,440,87.65
spending_behavior_education,438,87.25


---

## 5. Cleaning

The cleaning phase addresses 15 distinct issues found in the raw data. Every change is logged to an `issues_log` list, which is printed at the end for full auditability.

The cleaned file is saved separately from the raw file — **the raw file is never modified**.

In [54]:
# ── Config ─────────────────────────────────────────────────────────────────────
INPUT_PATH  = '/workspaces/dego-project-team11/data/raw_credit_applications.csv'
OUTPUT_PATH = os.path.join(os.path.dirname(INPUT_PATH), 'cleaned_credit_applications.csv')

# zip_code is loaded as string to preserve leading zeros (e.g. "01234" not 1234)
df_work = pd.read_csv(INPUT_PATH, dtype={'applicant_info_zip_code': str})
print(f"Loaded {len(df_work)} rows × {len(df_work.columns)} columns")

issues_log = []  # every cleaning action is recorded here

Loaded 502 rows × 36 columns


### Step 1 — Remove Duplicates

Three types of duplicates are addressed, in order of priority:

1. **Notes-flagged duplicates**: Some records have `DUPLICATE_ENTRY_ERROR` in the `decision_notes` field — an explicit marker from the upstream system. These are dropped first.
2. **Duplicate application IDs**: If two rows share the same `applicant_info_id`, only the first occurrence is kept.
3. **Duplicate SSNs**: Two applications with the same Social Security Number indicate the same individual applied more than once. Only the first record is retained.

The index is reset after removal to ensure clean integer indexing.

In [55]:
# Flag rows explicitly marked as duplicate in notes
duplicate_notes = df_work['decision_notes'].str.upper().str.contains('DUPLICATE', na=False)
df_work = df_work[~duplicate_notes]
issues_log.append(f"[Duplicates] Removed {duplicate_notes.sum()} rows flagged as DUPLICATE_ENTRY_ERROR in notes")

# Remove duplicate application IDs (keep first occurrence)
dup_ids = df_work['applicant_info_id'].duplicated(keep='first')
issues_log.append(f"[Duplicates] Removed {dup_ids.sum()} rows with duplicate applicant_info_id")
df_work = df_work[~dup_ids]

# Remove duplicate SSNs (keep first occurrence)
dup_ssns = df_work['applicant_info_ssn'].notna() & df_work['applicant_info_ssn'].duplicated(keep='first')
issues_log.append(f"[Duplicates] Removed {dup_ssns.sum()} rows with duplicate SSN")
df_work = df_work[~dup_ssns]

df_work = df_work.reset_index(drop=True)

### Step 2 — Standardise Gender

The raw data contains both abbreviated (`'M'`, `'F'`) and full (`'Male'`, `'Female'`) gender values, creating two separate categories for the same group. A simple mapping dictionary unifies all entries to the full-word format.

Before/after value counts are logged to confirm the transformation worked correctly.

In [56]:
gender_map = {'M': 'Male', 'F': 'Female'}
before = df_work['applicant_info_gender'].value_counts(dropna=False).to_dict()
df_work['applicant_info_gender'] = df_work['applicant_info_gender'].replace(gender_map)
after = df_work['applicant_info_gender'].value_counts(dropna=False).to_dict()
issues_log.append(f"[Gender] Standardized abbreviations — before: {before} | after: {after}")

### Step 3 — Validate Email Addresses

Emails are validated against a basic regex pattern: `local@domain.tld`. Any non-null value that fails this check is **nulled out** rather than dropped — the rest of the applicant's record may still be valid and useful.

The regex used: `^[\w\.-]+@[\w\.-]+\.\w{2,}$`

This catches common corruption patterns like missing `@`, missing domain, or placeholder strings like `"N/A"` or `"none"`.

In [57]:
email_pattern = r'^[\w\.-]+@[\w\.-]+\.\w{2,}$'
invalid_mask = df_work['applicant_info_email'].notna() & \
              ~df_work['applicant_info_email'].str.match(email_pattern, na=False)
invalid_emails = df_work.loc[invalid_mask, 'applicant_info_email'].tolist()
df_work.loc[invalid_mask, 'applicant_info_email'] = np.nan
issues_log.append(f"[Email] Nulled {len(invalid_emails)} invalid addresses: {invalid_emails}")

### Step 4 — Hash PII Fields (SSN & IP Address)

Social Security Numbers and IP addresses are **Personally Identifiable Information (PII)**. To protect applicant privacy while retaining the ability to detect duplicates and link records, both fields are replaced with their **SHA-256 hashes** — a one-way cryptographic transformation.

Properties of this approach:
- The original value **cannot be recovered** from the hash.
- The same input **always produces the same hash**, so duplicate detection still works.
- `NaN` values are preserved (not hashed) to avoid creating false non-null entries.

> ⚠️ After this step, the raw SSN/IP values are no longer accessible in this dataset. Always treat the cleaned file accordingly.

In [58]:
def pseudonymize_pii(value):
    if pd.isna(value) or value == "":
        return value
    
    # Combine the value with the pseudonym and encode
    salted_value = str(value) + SALT
    
    # Return the SHA-256 hexadecimal hash
    return hashlib.sha256(salted_value.encode()).hexdigest()

# 3. Apply the function to the PII columns
# Targeting the nested fields from the schema
df_work['applicant_info.ssn_hashed'] = df_work['applicant_info_ssn'].apply(pseudonymize_pii)
df_work['applicant_info.email_hashed'] = df_work['applicant_info_email'].apply(pseudonymize_pii)
df_work['applicant_info.full_name_hashed'] = df_work['applicant_info_full_name'].apply(pseudonymize_pii)
df_work['applicant_info.ip_address_hashed'] = df_work['applicant_info_ip_address'].apply(pseudonymize_pii)

issues_log.append(f"[PII] Pseudonymized SSN, email, full name, and IP address using salted SHA-256 hashing")

### Step 5 — Clean Zip Codes

Zip codes are 5-digit strings, but several issues arise when the column is read from CSV:

- Pandas may read them as **floats** (e.g., `1234.0`), adding a `.0` suffix.
- Short codes like `"501"` (a valid New York zip) lose their **leading zeros**.

Fix: strip the `.0` suffix if present, trim whitespace, then zero-pad with `zfill(5)`. The placeholder string `'00nan'` (produced when `NaN` is zero-padded) is converted back to `NaN`.

In [59]:
df_work['applicant_info_zip_code'] = (
    df_work['applicant_info_zip_code']
    .str.replace(r'\.0$', '', regex=True)   # remove trailing .0 if read as float
    .str.strip()
    .str.zfill(5)                            # zero-pad to 5 digits
)
df_work['applicant_info_zip_code'] = df_work['applicant_info_zip_code'].replace('00nan', np.nan)
issues_log.append("[Zip Code] Converted to zero-padded 5-digit string")

### Step 6 — Parse and Validate Date of Birth

The `dob` field has inconsistent separator formats (`/` vs `-`), so separators are normalised before parsing. After parsing, two validity checks are applied:

1. **Unparseable strings** — any value that cannot be converted to a valid date (e.g., `"not-a-date"`) becomes `NaT`.
2. **Unrealistic ages** — applicants must be between **18 and 100 years old**. Records outside this range are nulled (not dropped), as other fields may still be valid.

The column is stored as a `YYYY-MM-DD` string for consistent formatting across environments.

In [60]:
# Normalize separators
df_work['applicant_info_dob'] = (
    df_work['applicant_info_dob']
    .str.replace('/', '-', regex=False)
)

# Parse to datetime; unparseable → NaT
df_work['applicant_info_dob'] = pd.to_datetime(
    df_work['applicant_info_dob'],
    errors='coerce'
)

invalid_date_count = df_work['applicant_info_dob'].isna().sum()
if invalid_date_count > 0:
    issues_log.append(f"[DOB] Nulled {invalid_date_count} rows with invalid date format")

# Check ages
today = pd.Timestamp.today()
age = (today - df_work['applicant_info_dob']).dt.days / 365.25

invalid_age_mask = df_work['applicant_info_dob'].notna() & ((age < 18) | (age > 100))
invalid_age_count = invalid_age_mask.sum()
if invalid_age_count > 0:
    issues_log.append(f"[DOB] Nulled {invalid_age_count} rows with unrealistic age (outside 18–100)")

df_work.loc[invalid_age_mask, 'applicant_info_dob'] = pd.NaT

# Store as ISO date string
df_work['applicant_info_dob'] = df_work['applicant_info_dob'].dt.strftime('%Y-%m-%d')

### Step 7 — Annual Income: Flag Zeros

An `annual_income` of exactly `0` is almost certainly a data entry error — it is economically implausible for a credit applicant. Rather than silently dropping these rows:

- A boolean **flag column** `financials_annual_income_flagged` is added so analysts can investigate.
- The zero value itself is **replaced with `NaN`** to prevent it from distorting statistics or model training.

The income is also rounded to the nearest integer — annual income figures are typically reported as whole numbers, and fractional cents add no analytical value.

In [61]:
# Round to nearest integer
df_work['financials_annual_income'] = df_work['financials_annual_income'].round(0).astype('Int64')

zero_income = df_work['financials_annual_income'] == 0
issues_log.append(f"[Income] Flagged {zero_income.sum()} rows with annual_income = 0")
df_work['financials_annual_income_flagged'] = zero_income
df_work.loc[zero_income, 'financials_annual_income'] = np.nan

### Step 8 — Credit History Months: Remove Negatives

A negative credit history duration is physically impossible. Any negative values are **nulled** and logged. The column is cast to **nullable integer** (`Int64`) to support both whole-number values and `NaN` in the same column — standard `int64` cannot hold `NaN` in pandas.

In [62]:
negative_history_mask = df_work['financials_credit_history_months'] < 0
issues_log.append(f"[Credit History] Nulled {negative_history_mask.sum()} rows with negative credit_history_months: "
                  f"{df_work.loc[negative_history_mask, 'financials_credit_history_months'].tolist()}")
df_work.loc[negative_history_mask, 'financials_credit_history_months'] = pd.NA

# Nullable integer to support NaN + whole numbers
df_work['financials_credit_history_months'] = df_work['financials_credit_history_months'].astype('Int64')

### Step 9 — Debt-to-Income Ratio: Cap at 1.0

Debt-to-income (DTI) is defined as `total_debt / annual_income`. By definition, it is a ratio between `0` and `1.0` — a DTI above `1.0` means total debt exceeds annual income, which is already extreme. Values **above 1.0** in this dataset are data quality errors (likely incorrect decimal placement) and are nulled.

In [63]:
dti_mask = df_work['financials_debt_to_income'] > 1.0
issues_log.append(f"[DTI] Nulled {dti_mask.sum()} rows with debt_to_income > 1.0: "
                  f"{df_work.loc[dti_mask, 'financials_debt_to_income'].tolist()}")
df_work.loc[dti_mask, 'financials_debt_to_income'] = np.nan

### Step 10 — Savings Balance: Remove Negatives & Flag Outliers

Two-pass treatment for savings balance:

**Pass 1 — Remove negatives:** A negative savings balance is impossible in this context (it would represent overdraft, which is tracked separately). Negative values are nulled.

**Pass 2 — Flag statistical outliers:** Extreme high values are flagged using the **IQR method with a 3× threshold**:
```
outlier threshold = Q3 + 3 × IQR
```
This is more conservative than the standard 1.5× threshold — only the most extreme values are flagged, which is appropriate for a right-skewed financial variable where genuinely wealthy applicants can exist. A boolean flag column is added; values are not removed, as very high savings balances are plausible.

In [64]:
# Null negative values
negative_savings_mask = df_work['financials_savings_balance'] < 0
issues_log.append(f"[Savings] Nulled {negative_savings_mask.sum()} rows with negative savings_balance: "
                  f"{df_work.loc[negative_savings_mask, 'financials_savings_balance'].tolist()}")
df_work.loc[negative_savings_mask, 'financials_savings_balance'] = pd.NA

# Flag statistical outliers (IQR × 3)
Q1 = df_work['financials_savings_balance'].quantile(0.25)
Q3 = df_work['financials_savings_balance'].quantile(0.75)
IQR = Q3 - Q1
outlier_mask = df_work['financials_savings_balance'] > Q3 + 3 * IQR
issues_log.append(f"[Savings] Flagged {outlier_mask.sum()} rows as savings_balance outliers "
                  f"(threshold: > {Q3 + 3 * IQR:,.0f}): "
                  f"{df_work.loc[outlier_mask, 'financials_savings_balance'].tolist()}")
df_work['financials_savings_balance_flagged'] = outlier_mask

df_work['financials_savings_balance'] = df_work['financials_savings_balance'].astype('Int64')

### Step 11 — Spending Behavior: Fill NaN with 0

When a spending category is absent from a record's `spending_behavior` list, the pivoted column for that category will be `NaN`. The correct interpretation is that the applicant had **zero spend** in that category — the category simply wasn't included in their record because it had no transactions.

All `NaN` values in spending columns are therefore replaced with `0`, then cast to `int` (no fractional spending amounts in this dataset).

In [65]:
spending_cols = [c for c in df_work.columns if c.startswith('spending_behavior_')]
df_work[spending_cols] = df_work[spending_cols].fillna(0).astype(int)
issues_log.append(f"[Spending] Filled NaN with 0 across {len(spending_cols)} spending columns")

### Step 12 — Processing Timestamp: Parse & Flag Future Dates

The `processing_timestamp` is parsed as a timezone-aware datetime (UTC). Records with a timestamp **in the future** relative to now are flagged as suspicious — they indicate either system clock errors or records pre-loaded for future processing.

Future-dated records are **not removed** but flagged with a boolean column, so analysts can decide how to handle them. The timestamp is then formatted back to a string for clean CSV storage.

In [66]:
df_work['processing_timestamp'] = pd.to_datetime(df_work['processing_timestamp'], errors='coerce', utc=True)
future_mask = df_work['processing_timestamp'].notna() & (df_work['processing_timestamp'] > pd.Timestamp.now(tz='UTC'))
issues_log.append(f"[Timestamp] Flagged {future_mask.sum()} rows with future processing_timestamp")
df_work['processing_timestamp_flagged'] = future_mask
df_work['processing_timestamp'] = df_work['processing_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

### Step 13 — Decision Fields: Fill NaN with Context-Aware Labels

The `decision` fields have a natural logical relationship with `loan_approved`:

| Field | When `loan_approved = True` | When `loan_approved = False` |
|---|---|---|
| `rejection_reason` | Should be `N/A - Approved` | Unknown if missing |
| `interest_rate` | Unknown if missing | Should be `N/A - Rejected` |
| `approved_amount` | Unknown if missing | Should be `N/A - Rejected` |

Rather than leaving these as `NaN` (which is ambiguous), we fill them with **meaningful string labels** that preserve the logical context. This makes the data self-explanatory for downstream consumers.

`loan_purpose` is genuinely optional and has no logical constraint, so it is simply filled with `'Not Specified'`.

In [67]:
# rejection_reason is N/A when loan was approved
df_work['decision_rejection_reason'] = df_work['decision_rejection_reason'].where(
    df_work['decision_rejection_reason'].notna(),
    other=df_work['decision_loan_approved'].map({True: 'N/A - Approved', False: 'Unknown'})
)

# interest_rate and approved_amount are N/A when loan was rejected
df_work['decision_interest_rate'] = df_work['decision_interest_rate'].where(
    df_work['decision_interest_rate'].notna(),
    other=df_work['decision_loan_approved'].map({True: 'Unknown', False: 'N/A - Rejected'})
)
df_work['decision_approved_amount'] = df_work['decision_approved_amount'].where(
    df_work['decision_approved_amount'].notna(),
    other=df_work['decision_loan_approved'].map({True: 'Unknown', False: 'N/A - Rejected'})
)

# loan_purpose — genuinely optional
df_work['decision_loan_purpose'] = df_work['decision_loan_purpose'].fillna('Not Specified')

issues_log.append("[Decision] Filled NaNs with context-aware labels based on loan_approved status")

### Step 14 — Drop Low-Value Columns

The `decision_notes` column is dropped because:
- ~99.6% of values are `NaN` — it carries almost no information.
- The two non-null values were `DUPLICATE_ENTRY_ERROR` records already actioned in Step 1.

Dropping it avoids confusion for downstream users who might wonder why the column exists.

In [68]:
df = df_work.drop(columns=['decision_notes'])
issues_log.append("[Cleanup] Dropped decision_notes column (99.6% null, content already actioned)")

### Step 15 — Enforce Schema Types

After all transformations, column types are explicitly cast to their intended schema types. This prevents silent type coercion bugs when the CSV is re-read by other tools.

| Type | Columns | Notes |
|---|---|---|
| `str` | All `applicant_info_*`, decision text fields | `nan`/`None` strings converted back to `NaN` |
| `Int64` (nullable int) | `credit_history_months`, `savings_balance` | Supports `NaN` alongside integers |
| `float` | `annual_income`, `debt_to_income`, `interest_rate`, `approved_amount` | Preserves decimal precision |
| `bool` | `decision_loan_approved` | Explicit boolean |

> **Why nullable integer (`Int64`) instead of `int64`?**  
> Standard `int64` in pandas cannot represent `NaN`. `Int64` (capital I) is pandas' nullable integer type that supports both whole numbers and missing values.

In [69]:
# --- Strings ---
string_cols = [
    'applicant_info_id', 'applicant_info_full_name', 'applicant_info_first_name',
    'applicant_info_last_name', 'applicant_info_email', 'applicant_info_ssn',
    'applicant_info_ip_address', 'applicant_info_gender', 'applicant_info_dob',
    'applicant_info_zip_code',
    'decision_rejection_reason', 'decision_loan_purpose'
]
for col in string_cols:
    df_work[col] = df_work[col].astype(str).replace({'nan': np.nan, 'None': np.nan})

# --- Nullable integers ---
int_cols = ['financials_credit_history_months', 'financials_savings_balance']
for col in int_cols:
    df_work[col] = pd.to_numeric(df_work[col], errors='coerce').astype('Int64')

# --- Floats ---
float_cols = [
    'financials_annual_income', 'financials_debt_to_income',
    'decision_interest_rate', 'decision_approved_amount',
]
for col in float_cols:
    df_work[col] = pd.to_numeric(df_work[col], errors='coerce').astype(float)

# --- Boolean ---
df_work['decision_loan_approved'] = df_work['decision_loan_approved'].astype(bool)

issues_log.append("[Schema] Enforced correct dtypes across all columns per schema definition")

---

## 6. Export Cleaned Dataset

The cleaned DataFrame is saved to a new CSV file. The `issues_log` is printed as a full audit trail of every transformation applied.

In [70]:
df_work.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print(f"\n✓ Cleaned dataset saved: {len(df_work)} rows × {len(df_work.columns)} columns")
print(f"  → {OUTPUT_PATH}")
print("\n─── Issues Log ───────────────────────────────────────────────────────")
for entry in issues_log:
    print(" •", entry)


✓ Cleaned dataset saved: 498 rows × 43 columns
  → /workspaces/dego-project-team11/data/cleaned_credit_applications.csv

─── Issues Log ───────────────────────────────────────────────────────
 • [Duplicates] Removed 1 rows flagged as DUPLICATE_ENTRY_ERROR in notes
 • [Duplicates] Removed 1 rows with duplicate applicant_info_id
 • [Duplicates] Removed 2 rows with duplicate SSN
 • [Gender] Standardized abbreviations — before: {'Male': 193, 'Female': 193, 'F': 58, 'M': 52, nan: 2} | after: {'Female': 251, 'Male': 245, nan: 2}
 • [Email] Nulled 4 invalid addresses: ['mike johnson@gmail.com', 'test.user.outlook.com', 'john.doe@invalid', 'sarah.smith@']
 • [PII] Pseudonymized SSN, email, full name, and IP address using salted SHA-256 hashing
 • [Zip Code] Converted to zero-padded 5-digit string
 • [DOB] Nulled 105 rows with invalid date format
 • [Income] Flagged 1 rows with annual_income = 0
 • [Credit History] Nulled 2 rows with negative credit_history_months: [-10, -3]
 • [DTI] Nulled 1 

---

## 7. Post-Cleaning Inspection

Final sanity check on the cleaned dataset — descriptive statistics and variable types should now be consistent with the schema and free of the anomalies identified during EDA.

In [71]:
print("\nDescriptive Statistics for Numerical Columns after cleaning:")
print(df_work.describe())

print("\nDescriptive Statistics for Non-Numerical Columns after cleaning:")
print(df_work.describe(include='object'))


Descriptive Statistics for Numerical Columns after cleaning:
       financials_annual_income  financials_credit_history_months  \
count                492.000000                             496.0   
mean               82752.912602                         50.612903   
std                27910.968499                         31.188889   
min                22000.000000                               0.0   
25%                63000.000000                             27.75   
50%                81000.000000                              48.5   
75%               101000.000000                              72.0   
max               171000.000000                             133.0   

       financials_debt_to_income  financials_savings_balance  \
count                 497.000000                       497.0   
mean                    0.241932                29516.022133   
std                     0.115793                16522.124074   
min                     0.050000                         0.0

/tmp/ipykernel_2826/3667105686.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df_work.describe(include='object'))


In [72]:
# Types of variables

numerical_variables = df_work.select_dtypes(include=[np.number]).columns.tolist()
print("\nNumerical Variables:", numerical_variables)

categorical_variables = df_work.select_dtypes(include=['object']).columns.tolist()
print("\nCategorical Variables:", categorical_variables)


Numerical Variables: ['financials_annual_income', 'financials_credit_history_months', 'financials_debt_to_income', 'financials_savings_balance', 'spending_behavior_adult_entertainment', 'spending_behavior_alcohol', 'spending_behavior_dining', 'spending_behavior_education', 'spending_behavior_entertainment', 'spending_behavior_fitness', 'spending_behavior_gambling', 'spending_behavior_groceries', 'spending_behavior_healthcare', 'spending_behavior_insurance', 'spending_behavior_rent', 'spending_behavior_shopping', 'spending_behavior_transportation', 'spending_behavior_travel', 'spending_behavior_utilities', 'decision_interest_rate', 'decision_approved_amount']

Categorical Variables: ['applicant_info_id', 'applicant_info_full_name', 'applicant_info_first_name', 'applicant_info_last_name', 'applicant_info_email', 'applicant_info_ssn', 'applicant_info_ip_address', 'applicant_info_gender', 'applicant_info_dob', 'applicant_info_zip_code', 'decision_rejection_reason', 'decision_loan_purpose'

/tmp/ipykernel_2826/3112851084.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_variables = df_work.select_dtypes(include=['object']).columns.tolist()
